# Preparation project 5th semester 

The following contents of this document are intended to give an overview of the source data for a possible project. The goal is to develop, through analysis of this data, potential research questions that can serve as the basis for a project. The source data was provided on 16.07.2025 10:11 by Maximilian Lackner in the MS Teams chat "Energie Projekt".



## 1. Overview of the datasets

The source data consists of two datasets: 

- weather_data_ratzersdorf.csv

- metering_point_data.csv

The dataset "weather_data_ratzersdorf.csv" provides information on the weather over a certain period in Ratzersdorf (a district of St. Pölten, Lower Austria). 

The dataset "metering_point_data.csv" describes the electricity consumption and feed-in data recorded at specific metering points of an energy community with several members in Ratzersdorf over a certain period.  

## 2. Libraries used 

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import seaborn as sns
from sklearn.utils import shuffle
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, make_scorer, f1_score, accuracy_score, confusion_matrix, classification_report
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier


## 3. Importing the datasets

In [ ]:
weather_df = pd.read_csv("./weather_data_ratzersdorf.csv")

metering_point_df = pd.read_csv("./metering_point_data.csv")

## 4. Data analysis

The following sections provide a separate description of the datasets. 

### 4.1. Weather data

#### 4.1.1. General 

The dataset contains hourly measured weather data, which can typically be used as external influencing factors in energy consumption or production models. 

| Feature                                      | Description                                                                                |
| -------------------------------------------- | ------------------------------------------------------------------------------------------- |
| **`time`**                                   | Measurement timestamp (UTC time)                                                          |
| **`air_temperature [degree_Celsius]`**       | Air temperature in degrees Celsius                                                              |
| **`global_radiation [W m⁻²]`**               | Global solar radiation on the horizontal surface, in watts per square metre             |
| **`humidity [percent]`**                     | Relative humidity in percent                                                        |
| **`wind_speed_eastward [m s⁻¹]`**            | Eastward component of wind speed (positive = easterly wind, negative = westerly wind)              |
| **`wind_speed_northward [m s⁻¹]`**           | Northward component of wind speed (positive = northerly wind, negative = southerly wind)             |
| **`mean_sea_level_pressure [Pa]`**           | Air pressure at sea level, in pascal                                                         |
| **`1_hour_precipitation_sum [kg m⁻²]`**      | Precipitation amount within the last hour, in kilograms per square metre (≈ mm of rain) |
| **`dew_point_temperature [degree_Celsius]`** | Dew point temperature in degrees Celsius (the temperature at which air becomes saturated)                   |


In [ ]:
weather_df.head(5)

The dataset has 13,105 entries and nine features. Some features have missing entries. 

In [ ]:
# Convert the timestamp to a date format
weather_df["time"] = pd.to_datetime(weather_df["time"])

The statistical properties can be seen from the following table:

In [ ]:
weather_df.describe()

#### 4.1.2. Plausibility of the data

The following section analyses the data distribution and the occurrence of missing values. Seasonal effects are expected to produce noticeable correlations between individual weather features.

#### 4.1.2.1 Completeness of the time entries

The dataset is complete over time, i.e. no entries are missing.

In [ ]:
print("First entry:")
weather_df.head(1)

In [ ]:
print("\nLast entry:")
weather_df.tail(1)

In [ ]:
start = weather_df['time'].min()
end = weather_df['time'].max()

expected_hour_count = int((end - start) / pd.Timedelta(hours=1)) + 1
actual_count = len(weather_df)

print(f"Expected number of hours: {expected_hour_count}")
print(f"Actual number of entries: {actual_count}")

#### 4.1.2.2 Missing entries

The dataset contains some missing entries, which however affect only nine days. It would be worthwhile to investigate these gaps more closely, since several missing values often occur together. It is possible that maintenance work was carried out during these periods, or that disruptions occurred due to technical faults. 

In [ ]:
# count the number of missing values per row
weather_df['missing_count'] = weather_df.isna().sum(axis=1)

# show rows with more than one missing value
multi_missing = weather_df[weather_df['missing_count'] > 1]
multi_missing = multi_missing.sort_values(by='time', ascending=False)
multi_missing

In [ ]:
# Ensure 'date' is present (otherwise extract it)
multi_missing = multi_missing.copy()
multi_missing['date'] = multi_missing['time'].dt.date

# Unique days with multiple missing values
affected_days = multi_missing['date'].unique()

sorted(affected_days)

#### 4.1.2.3 Visualisation of the features

The following charts reveal the following patterns:

- The average daily temperature varies noticeably with the seasons.

- The average global solar radiation on a horizontal surface (watts per square metre per day) also shows a pronounced seasonal dependency.

- The average daily humidity also appears to be subject to seasonal variation.

- The average daily eastward and northward components of wind speed, on the other hand, appear to be independent of the season.

- The average air pressure at sea level (pascal per day) shows stronger outliers in both directions, particularly during the colder seasons.

- The average precipitation amount within the last hour (kilograms per square metre, roughly equivalent to mm of rain) affects individual periods, which can be used to identify rainy days. No clear correlation with other features is apparent.

- The average dew point temperature in degrees Celsius (the temperature at which humidity condenses) appears to correlate with the average daily temperature.

The "missing_count" feature indicates the distribution of NaN entries. 

In [ ]:
import matplotlib.pyplot as plt

# extract date
weather_df['date'] = weather_df['time'].dt.date

# select only numeric columns (except 'time'/'date')
numeric_columns = weather_df.select_dtypes(include='number').columns.tolist()

# compute daily averages
daily_avg = weather_df.groupby('date')[numeric_columns].mean()

# compute 7-day rolling average
rolling_avg = daily_avg.rolling(window=7, min_periods=1).mean()

# create plots
plt.figure(figsize=(15, 4 * len(numeric_columns)))  # adjust height per plot

for i, column in enumerate(numeric_columns, 1):
    plt.subplot(len(numeric_columns), 1, i)
    
    # original daily average
    plt.plot(daily_avg.index, daily_avg[column], label=f'Daily average {column}', color='blue')
    
    # 7-day average (red)
    plt.plot(rolling_avg.index, rolling_avg[column], label='7-day average', color='red', linewidth=2)
    
    plt.title(f'{column}: daily average & 7-day average')
    plt.xlabel('Date')
    plt.ylabel(column)
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.legend()

plt.tight_layout()
plt.show()


#### 4.1.2.4 Visualisation of the correlations

– The correlation matrix shows that air temperature is moderately related to global solar radiation on a horizontal surface, and strongly related to humidity. A very strong correlation exists with dew point temperature (in degrees Celsius).

– Global solar radiation correlates moderately with air temperature and strongly with humidity.

– Humidity shows a strong correlation with both air temperature and global solar radiation.

– The eastward component of wind speed correlates very strongly with the northward component – and vice versa.

– Air pressure at sea level is moderately related to air temperature and dew point temperature.

– The precipitation amount within the last hour (in kilograms per square metre, roughly equivalent to mm of rain) shows no significant correlation with other features.

– Dew point temperature, in turn, correlates very strongly with air temperature and moderately with air pressure at sea level.



Considerations for feature selection: 

- Strongly correlated features such as air temperature, humidity, and dew point temperature should be reduced to avoid redundancy.

- Independent features such as precipitation amount contain distinct information and should be kept in the model.

- Avoiding multicollinearity is particularly critical for linear models; for tree-based models this is less critical.

In [ ]:
# select only numeric columns
numeric_columns = weather_df.select_dtypes(include='number').columns

# compute correlation matrix (Pearson correlation)
corr_matrix = weather_df[numeric_columns].corr()

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', cbar=True,
            square=True, linewidths=0.5, linecolor='gray')
plt.title('Correlation matrix of the numeric features')
plt.show()

The scatter plots do not reveal any non-linear or complex relationships beyond the linear relationships already shown in the Pearson correlation matrix.

In [ ]:
import seaborn as sns
"""""
# Pairplot without density curves
pairplot = sns.pairplot(
    weather_df[numeric_columns],
    height=2.5,
    diag_kind=None,
    plot_kws={'s': 15, 'alpha': 0.6}
)

# Save instead of displaying
pairplot.savefig("scatterplots_all_features.png", dpi=300, bbox_inches='tight')
"""""

### 4.2. Metering point data

#### 4.2.1. General



The dataset contains measurements of energy consumption and generation, supplemented with communication parameters and the direction of the energy flow.

Source: https://docs.eegfaktura.at/books/workflowprozesse/page/energiedaten-herunterladen

| Feature               | Description / possible meaning                                                                                                           |
| --------------------- | ------------------------------------------------------------------------------------------------------------------------------------------- |
| **obj\_id**           | Unique object number or ID                                                                                                             |
| **time**              | Timestamp (date and time, in UTC)                                                                                                     |
| **period\_interval**  | Measurement time interval, e.g. 15 minutes                                                                                                 |
| **energy\_direction** | Direction of the energy flow, e.g. "C" for Consumption or "G" for Generation                                       |
| **wt\_meas\_cons**    | Actually measured consumption of the metering point, already weighted by the participation factor if the metering point is in several RECs             |
| **comm\_pot**         | Energy potential from the energy community that would have been available to the metering point (can be greater or less than consumption) |
| **comm\_cov**         | Consumption actually covered by the energy community; corresponds either to the total generation or the available potential    |
| **wt\_meas\_gen**     | Total energy produced by the metering point, already weighted by the participation factor                                                          |
| **wt\_surp\_gen**     | Surplus produced energy that is not needed within the energy community and is therefore exported to the grid           |



From the first entries of the dataset it can be seen that no energy was generated, so comm_pot and comm_cov are zero, and the energy was therefore supplied externally. 

In [ ]:
metering_point_df.head(10)

The dataset has 8,858,880 entries and nine features. Some features have missing entries. Based on obj_id, it can be seen that 306 producers and/or consumers are contained in the dataset.

An obj_id describes either a consumer or a producer – both at the same time is not possible. It can be assumed that most households have two or more obj_ids. However, a unique mapping of obj_ids to individual households is not possible based on the available dataset.

In [ ]:
# example: filter for obj_id = 673
target_obj_id = 673
filtered_data = metering_point_df[metering_point_df['obj_id'] == target_obj_id]
filtered_data["energy_direction"].unique()


In [ ]:
metering_point_df.groupby('obj_id')['energy_direction'].nunique().value_counts()

#### 4.2.2. Plausibility of the data

The dataset contains some missing entries, which however affect only the features wt_meas_cons, comm_pot, comm_cov, wt_meas_gen, and wt_surp_gen. From the examples it can be seen that for Consumption, wt_meas_gen and wt_surp_gen take NaN values, since no values were recorded here. 

In [ ]:
metering_point_df.sample(10)

#### 4.2.2.2 Missing entries

The number of entries per object ID (obj_id) varies; some objects have up to 52,413 entries, while others have only 1,065 entries. 

In [ ]:
# compute the number of entries per obj_id
metering_point_df['obj_id'].value_counts()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# count entries per obj_id
obj_counts = metering_point_df['obj_id'].value_counts().reset_index()
obj_counts.columns = ['obj_id', 'count']

# define classes (adjust as needed!)
bins = [0, 1000, 2000, 5000, 10000, 20000, 50000, 100000, 200000]
labels = [
    '1–1000', 
    '1001–2000', 
    '2001–5000', 
    '5001–10000', 
    '10001–20000', 
    '20001–50000', 
    '50001–100000', 
    '100001–200000'
]
# assign classes
obj_counts['class'] = pd.cut(obj_counts['count'], bins=bins, labels=labels, right=True, include_lowest=True)

# count the number of obj_ids per class
class_counts = obj_counts['class'].value_counts().sort_index()

# plot
plt.figure(figsize=(12, 5))
plt.bar(class_counts.index.astype(str), class_counts.values, color='steelblue')

# title and axis labels
plt.title('Distribution of obj_id by number of entries (classified)')
plt.xlabel('Classes of number of entries')
plt.ylabel('Number of obj_id')

# show values above the bars
for i, v in enumerate(class_counts.values):
    plt.text(i, v + 0.3, str(v), ha='center', va='bottom')

plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


The following chart shows that not all metering points have been present since the beginning. It can be assumed that some metering points were added or removed during the observation period. 

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 🔹 select 20 random obj_ids
unique_ids = metering_point_df['obj_id'].unique()
sample_ids = np.random.choice(unique_ids, size=min(20, len(unique_ids)), replace=False)

# 🔹 filter data for these IDs
df_sample = metering_point_df[metering_point_df['obj_id'].isin(sample_ids)].copy()
df_sample['time'] = pd.to_datetime(df_sample['time'])

# 🔹 sort by obj_id and time
df_sample = df_sample.sort_values(['obj_id', 'time'])

# 🔹 prepare plot
plt.figure(figsize=(12, 8))

# 🔹 plot each obj_id at its own y position
for i, obj_id in enumerate(sorted(sample_ids)):
    df_obj = df_sample[df_sample['obj_id'] == obj_id]
    plt.plot(df_obj['time'], [i]*len(df_obj), '|', markersize=10, label=str(obj_id))

# 🔹 axes & layout
plt.title('Measurement timestamps for 20 random obj_id')
plt.xlabel('Time')
plt.yticks(range(len(sample_ids)), sorted(sample_ids))
plt.ylabel('obj_id')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


The chart below shows that the available energy fluctuates daily; at times there is a surplus, while at other times of day an under-coverage must occur. The reason for this can be the type of energy producers.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# 🔹 ensure timestamps are in the correct format
metering_point_df['time'] = pd.to_datetime(metering_point_df['time'])

# 🔹 period 01.05.2024 – 30.05.2024
mask = (metering_point_df['time'] >= '2024-05-01') & (metering_point_df['time'] <= '2024-05-30')
df_period = metering_point_df.loc[mask].copy()

# 🔹 only valid wt_surp_gen values
df_period = df_period.dropna(subset=['wt_surp_gen'])

# 🔹 sum values per timestamp
df_sum = df_period.groupby('time', as_index=False)['wt_surp_gen'].sum()

# 🔹 sort by time
df_sum = df_sum.sort_values('time')

# 🔹 plot
plt.figure(figsize=(12, 6))
plt.plot(df_sum['time'], df_sum['wt_surp_gen'], color='steelblue', linewidth=1.8)

# 🔹 title & axes
plt.title('Total surplus energy generation (wt_surp_gen) – 01.05.2024 to 30.05.2024')
plt.xlabel('Time')
plt.ylabel('Sum wt_surp_gen')

# 🔹 fine grid & layout
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()
